# intraday-price-delay-v3

BigAlpha 2026 · AI 因子挖掘 (AI track) submission.

- source: `platform/submission/intraday_price_delay_v3_submit.py`
- sha1: `600c1dc35eea36eac3a77024caddbc1ab8c6c43e`
- generated by `platform/submission/build_notebook.py` — the code cell below is a **verbatim copy** of that committed file, so this notebook contains the code that is in git.

AI-track note: the derivation chain for this factor (prompts, agent sessions, where the work departed from its source) is in `factors/intraday-price-delay-v3/provenance.md`, and the AI application document is submitted alongside as markdown.


In [ ]:
"""intraday-price-delay-v3 (C2) — submission main. Five-gate chain complete (k44/k45/k46).

pc_k = pcorr(r_i, r_m(t-k) | r_m) for k=1..3 with exp weights e^{-0.5(k-1)}; a lag counts only
when the session-minute index gap is exactly k (lunch break invalidates cross-break lags);
market = leave-one-out equal-weight minute mean; r_i clipped to +-10%; D = sum(w_k pc_k^2) /
(rho_0^2 + sum(w_k pc_k^2)) from 5-DAY POOLED sufficient statistics; factor = -z_by_day(D).

Same latent trait as the live `delay` (board 0.657-0.724, A 0.1233 / B 0.8938 in the 07-30 daily
report -> 94% of that score is orthogonality), cleaner estimator: the contemporaneous-market
partial correlation removes the market-autocorrelation artifact, LOO removes the own-stock term,
and 5-day pooling fixes the measured estimation-noise defect (daily AC1 0.128 -> 0.809).
Measured: k44 discovery corr vs live v1 = 0.347 (under the 0.40 near-duplicate cap);
k45/k46 official evaluator neutralized ic_ir -0.14637 vs the live factor's -0.10177 (+43.8%),
identical to 16 digits on the window intersection; [v1 + 7 canon] replicate only 19.9% of its
per-day variance. NEGATED orientation declared before submission: raw sign is - three ways, the
evaluator's A rewards positive IC, and B is sign-invariant.

Contract deviations, both stated rather than hidden: (1) no bigalpha_2026_instruments join
(datasources exposes bar1m + financial only), so the cross-section is every bar1m name -- z over
a superset preserves ranking inside the evaluator's universe; (2) the query window extends 15
calendar days BACKWARD for the 5-day pooling warmup -- if the platform hard-clips it, the first
~2 output days degrade to all-neutral (every day still present, coverage 1.0), measured in a
stub dryrun. Unmeasurable name-day -> z = 0 (neutral), never a deleted day. PIT: minute rows of
days <= t only; sqrt inputs clipped at 0; no assertions, no raising.
"""


def main(datasources, start_date, end_date):
    import numpy as np
    import pandas as pd
    import dai

    MIN_MIN_DAY = 120       # per-day minutes floor for rho0 (pre-pooling this is per-day n0)
    MIN_MIN_LAG = 100       # per-lag valid-pair floor, applied to the POOLED counts
    WK = [1.0, 0.6065306597126334, 0.36787944117144233]
    WARMUP_DAYS = 15        # calendar days queried before start_date, dropped from the output

    SQL_TEMPLATE = """
    WITH r AS (
        SELECT instrument, date AS ts, CAST(date AS DATE) AS d,
               EXTRACT(HOUR FROM date) * 60 + EXTRACT(MINUTE FROM date) AS mins,
               close
        FROM {bar1m}
    ),
    w AS (
        SELECT instrument, d, ts,
               CASE WHEN mins <= 690 THEN mins - 571 ELSE mins - 781 + 201 END AS idx,
               CASE WHEN close > 0
                     AND lag(close) OVER (PARTITION BY instrument, d ORDER BY ts) > 0
                    THEN GREATEST(-0.10, LEAST(0.10,
                         close / lag(close) OVER (PARTITION BY instrument, d ORDER BY ts) - 1))
               END AS ri
        FROM r
        WHERE (mins BETWEEN 571 AND 690) OR (mins BETWEEN 781 AND 896)
    ),
    m1 AS (
        SELECT instrument, d, ts, idx, ri,
               COUNT(ri) OVER (PARTITION BY d, ts) AS nmkt,
               SUM(ri)   OVER (PARTITION BY d, ts) AS smkt
        FROM w
    ),
    m2 AS (
        SELECT instrument, d, ts, idx, ri,
               CASE WHEN nmkt > 1 AND ri IS NOT NULL
                    THEN (smkt - ri) / (nmkt - 1) END AS rm
        FROM m1
    ),
    j AS (
        SELECT instrument, d, idx, ri, rm,
               lag(rm, 1)  OVER (PARTITION BY instrument, d ORDER BY ts) AS rmL1,
               lag(rm, 2)  OVER (PARTITION BY instrument, d ORDER BY ts) AS rmL2,
               lag(rm, 3)  OVER (PARTITION BY instrument, d ORDER BY ts) AS rmL3,
               lag(idx, 1) OVER (PARTITION BY instrument, d ORDER BY ts) AS ix1,
               lag(idx, 2) OVER (PARTITION BY instrument, d ORDER BY ts) AS ix2,
               lag(idx, 3) OVER (PARTITION BY instrument, d ORDER BY ts) AS ix3
        FROM m2
    ),
    v AS (
        SELECT instrument, d, ri, rm,
               CASE WHEN idx - ix1 = 1 THEN rmL1 END AS m1v,
               CASE WHEN idx - ix2 = 2 THEN rmL2 END AS m2v,
               CASE WHEN idx - ix3 = 3 THEN rmL3 END AS m3v
        FROM j
        WHERE ri IS NOT NULL AND rm IS NOT NULL
    )
    SELECT d AS date, instrument,
           COUNT(*) AS n0,
           SUM(ri) AS si0, SUM(ri * ri) AS sii0,
           SUM(rm) AS sm0, SUM(rm * rm) AS smm0, SUM(ri * rm) AS sim0,
           COUNT(m1v) AS n1, SUM(CASE WHEN m1v IS NOT NULL THEN ri END) AS si1,
           SUM(CASE WHEN m1v IS NOT NULL THEN ri * ri END) AS sii1,
           SUM(CASE WHEN m1v IS NOT NULL THEN rm END) AS sm01,
           SUM(CASE WHEN m1v IS NOT NULL THEN rm * rm END) AS smm01,
           SUM(m1v) AS sk1, SUM(m1v * m1v) AS skk1,
           SUM(CASE WHEN m1v IS NOT NULL THEN ri * rm END) AS sim01,
           SUM(ri * m1v) AS sik1, SUM(rm * m1v) AS smk1,
           COUNT(m2v) AS n2, SUM(CASE WHEN m2v IS NOT NULL THEN ri END) AS si2,
           SUM(CASE WHEN m2v IS NOT NULL THEN ri * ri END) AS sii2,
           SUM(CASE WHEN m2v IS NOT NULL THEN rm END) AS sm02,
           SUM(CASE WHEN m2v IS NOT NULL THEN rm * rm END) AS smm02,
           SUM(m2v) AS sk2, SUM(m2v * m2v) AS skk2,
           SUM(CASE WHEN m2v IS NOT NULL THEN ri * rm END) AS sim02,
           SUM(ri * m2v) AS sik2, SUM(rm * m2v) AS smk2,
           COUNT(m3v) AS n3, SUM(CASE WHEN m3v IS NOT NULL THEN ri END) AS si3,
           SUM(CASE WHEN m3v IS NOT NULL THEN ri * ri END) AS sii3,
           SUM(CASE WHEN m3v IS NOT NULL THEN rm END) AS sm03,
           SUM(CASE WHEN m3v IS NOT NULL THEN rm * rm END) AS smm03,
           SUM(m3v) AS sk3, SUM(m3v * m3v) AS skk3,
           SUM(CASE WHEN m3v IS NOT NULL THEN ri * rm END) AS sim03,
           SUM(ri * m3v) AS sik3, SUM(rm * m3v) AS smk3
    FROM v
    GROUP BY d, instrument
    ORDER BY d, instrument
    """
    sql = SQL_TEMPLATE.format(bar1m=datasources["bar1m"])

    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    if (end.hour, end.minute, end.second) == (0, 0, 0):
        end = end + pd.Timedelta(hours=23, minutes=59, seconds=59)
    q_start = start - pd.Timedelta(days=WARMUP_DAYS)
    filters = {"date": [str(q_start), str(end)]}

    df = dai.query(sql, filters=filters, compression=True).df()
    df["date"] = pd.to_datetime(df["date"])          # DAI returns datetime.date (trap D4)
    for c in df.columns:
        if c not in ("date", "instrument"):
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("float64")
    df = df.replace([np.inf, -np.inf], np.nan).sort_values(
        ["instrument", "date"]).reset_index(drop=True)

    sumcols = [c for c in df.columns if c not in ("date", "instrument")]
    g = df.groupby("instrument", sort=False)
    for c in sumcols:
        df[c] = g[c].transform(lambda s: s.rolling(5, min_periods=3).sum())

    def corr_from(n, sx, sxx, sy, syy, sxy):
        cov = sxy - sx * sy / n
        vx = (sxx - sx ** 2 / n).clip(lower=0.0)     # sqrt(negative) RAISES here (v1-main trap)
        vy = (syy - sy ** 2 / n).clip(lower=0.0)
        den = np.sqrt(vx * vy)
        return (cov / den.where(den > 0)).clip(-1.0, 1.0)

    r0 = corr_from(df["n0"].where(df["n0"] >= MIN_MIN_DAY), df["si0"], df["sii0"],
                   df["sm0"], df["smm0"], df["sim0"])
    num = pd.Series(0.0, index=df.index)
    for k, wk in zip((1, 2, 3), WK):
        nk = df[f"n{k}"].where(df[f"n{k}"] >= MIN_MIN_LAG)
        rik = corr_from(nk, df[f"si{k}"], df[f"sii{k}"],
                        df[f"sk{k}"], df[f"skk{k}"], df[f"sik{k}"])
        r0k = corr_from(nk, df[f"si{k}"], df[f"sii{k}"],
                        df[f"sm0{k}"], df[f"smm0{k}"], df[f"sim0{k}"])
        rmk = corr_from(nk, df[f"sm0{k}"], df[f"smm0{k}"],
                        df[f"sk{k}"], df[f"skk{k}"], df[f"smk{k}"])
        den = np.sqrt(((1 - r0k ** 2) * (1 - rmk ** 2)).clip(lower=1e-12))
        pc = ((rik - r0k * rmk) / den).clip(-1.0, 1.0)
        num = num + wk * pc ** 2
    tot = r0 ** 2 + num
    d5 = (num / tot.where(tot > 0)).replace([np.inf, -np.inf], np.nan)

    # NEGATED z per day; unmeasurable name-day -> neutral 0 (the general rule); warmup rows drop
    day = df["date"]
    gz = d5.groupby(day)
    z = (d5 - gz.transform("mean")) / gz.transform("std")
    factor = (-z).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    factor = factor.where(np.isfinite(factor))

    out = pd.DataFrame({"date": day, "instrument": df["instrument"],
                        "factor": factor.astype("float64")})
    out = out[out["date"] >= start]
    out = out.drop_duplicates(subset=["date", "instrument"], keep="first")
    return out.dropna(subset=["factor"]).sort_values(["date", "instrument"]).reset_index(drop=True)


In [ ]:
# The evaluation-module call must live in the notebook (competition requirement).
datasources = {
    'bar1m': 'bigalpha_2026_stock_bar1m',
    'financial': 'bigalpha_2026_financial',
}
START, END = '2019-06-05', '2023-12-31 23:59:59'

factor_data = main(datasources, START, END)
print('rows', len(factor_data), 'cols', list(factor_data.columns))
print('days', factor_data['date'].nunique(),
      'names', factor_data['instrument'].nunique())

try:
    from bigmodule import M
    result = M.bigalpha_eval._latest(factor_data=factor_data)
    print(dict(result.factor_analyze))
except Exception as exc:
    # Never raise from a submitted notebook: the executor returns no error detail, so a
    # raise is indistinguishable from a broken factor. Report and let the platform's own
    # scoring pass judge the returned frame.
    print('evaluator not run in this environment:', type(exc).__name__, exc)
